# Synthetic bootstrap-null calibration: c-GC*

This notebook calibrates the synthetic graph-instability statistic for **c-GC*** over `p=1,...,7` with the reported graph horizon fixed at one.

It is safe to run concurrently with `bootstrap_null_synthetic_c-GC.ipynb`: results and replicate checkpoints are stored in disjoint method directories.

Progress is shown at three levels: overall scenarios, bootstrap replicates, and active conditioning depths. Completion prints report `T_obs`, `critical_95`, the bootstrap `p_value`, reused and newly computed replicate counts, elapsed time, and output paths.

`B` is the **target cumulative replicate count**. Run with `B=5`, then set `B=10` to reuse the first five compatible checkpoints and compute only five additional replicates.

This notebook generates its own controlled synthetic samples. The simulation `extension_metrics.ipynb` notebooks are not prerequisites.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError('Could not find the project root')
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.synthetic_calibration import (
    run_synthetic_calibration,
)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Increase this cumulative target to extend a completed run.
B = 5
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
P0 = 1
BLOCK_LENGTH = 1
SEED = 42
N_JOBS = 1
T = 2000
D = 10
SHOW_PROGRESS = True

METHODS_TO_RUN = ['c-GC-star']
SCENARIOS_TO_RUN = [
    'order1_unconfounded',
    'order1_with_latent_confounder',
    'order3_unconfounded',
]
OUTPUT_DIR = (
    PROJECT_ROOT / 'outputs' / 'calibration' / 'synthetic' / 'c-GC-family'
)

maximum_depth_fits = (
    len(METHODS_TO_RUN)
    * len(SCENARIOS_TO_RUN)
    * (B + 1)
    * len(P_VALUES)
)
print('=' * 72)
print('Configured synthetic bootstrap job')
print(f'Method: {METHODS_TO_RUN[0]}')
print(f'Scenarios: {SCENARIOS_TO_RUN}')
print(f'Depths: {P_VALUES}; p0={P0}')
print(f'Cumulative target: B={B}; block_length={BLOCK_LENGTH}')
print(f'Data dimensions: T={T}, d={D}')
print(f'Maximum single-depth fits before checkpoint reuse: {maximum_depth_fits}')
print(f'Output root: {OUTPUT_DIR}')
print('=' * 72)

In [ ]:
artifacts = run_synthetic_calibration(
    PROJECT_ROOT,
    output_dir=OUTPUT_DIR,
    method_names=METHODS_TO_RUN,
    scenario_names=SCENARIOS_TO_RUN,
    p_values=P_VALUES,
    p0=P0,
    B=B,
    block_length=BLOCK_LENGTH,
    seed=SEED,
    T=T,
    d=D,
    n_jobs=N_JOBS,
    show_progress=SHOW_PROGRESS,
)

summary = artifacts['summary']
print('\nCompleted summary:')
print(summary.to_string(index=False))
summary

In [ ]:
required_columns = {
    'method',
    'scenario',
    'T_obs',
    'critical_95',
    'p_value',
    'B',
    'reused_replicates',
}
missing_columns = sorted(required_columns.difference(summary.columns))
if missing_columns:
    raise ValueError(f'Missing summary columns: {missing_columns}')
if set(summary['method']) != {'c-GC-star'}:
    raise ValueError(f'Unexpected methods: {sorted(set(summary["method"]))}')
if not summary['p_value'].between(0.0, 1.0).all():
    raise ValueError('One or more bootstrap p-values are outside [0, 1]')

print('\nOutput verification:')
for key in (
    'results_path',
    'summary_path',
    'null_plot_path',
    'pointwise_plot_path',
    'manifest_path',
):
    path = artifacts[key]
    if not path.exists():
        raise FileNotFoundError(path)
    print(f'✓ {key}: {path.relative_to(PROJECT_ROOT)}')

pd.read_csv(artifacts['summary_path'])